# Notebook 05 — Data Quality Validation

**Objective:** Comprehensive validation of the unified market dataset before downstream analysis.

**Checks:**
1. Schema validation (columns, data types)
2. Grain validation (Date × CODE_ISIN uniqueness)
3. Identifier validation (CODE_ISIN, Company)
4. Date validation
5. Price data quality
6. Volume data quality
7. Cross-dataset consistency
8. Actionable recommendations

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ingestion import ingest_workbook
from src.validation import validate_dataset

print(f'Project root: {ROOT}')

Project root: /home/yass/Desktop/DSS_CMR


## Step 1: Load and prepare dataset

In [2]:
wb_path = ROOT / 'Données Marché Boursier_Projet_IA_.xlsx'
required_vars = {'Cours', 'Bid', 'Ask', 'Volume MC', 'Quantité MC'}

print('Loading workbook...')
unified, ingest_report = ingest_workbook(str(wb_path), required_variables=required_vars)

print(f'\n✓ Dataset loaded:')
print(f'  Shape: {unified.shape}')
print(f'  Records: {ingest_report["unified_records"]}')
print(f'  Companies: {ingest_report["unified_companies"]}')
print(f'  Sessions: {ingest_report["unified_sessions"]}')

Loading workbook...
⊗ Excluded: Data -> Sheet type is family_b, not market Family A
✓ Included: Cours -> Cours (155791 records)
✓ Included: Bid -> Bid (47025 records)
✓ Included: Ask -> Ask (47025 records)
✓ Included: Quantité MC -> Quantité MC (155708 records)
✓ Included: Volume MC -> Volume MC (155708 records)
⊗ Excluded: Indicateurs -> Sheet type is unknown, not market Family A

✓ Dataset loaded:
  Shape: (141217, 8)
  Records: 141217
  Companies: 77
  Sessions: 1877


## Step 2: Run comprehensive validation

In [3]:
print('Running validation checks...\n')
all_passed, validation_report = validate_dataset(unified, verbose=True)

Running validation checks...


DATA QUALITY VALIDATION REPORT

SCHEMA
----------------------------------------------------------------------
  ✓ [info    ] Schema - All Columns: All required columns present
  ⚠ [warning ] Schema - Data Types: Date: got datetime64[us], expected datetime64[ns]; CODE_ISIN: got str, expected object; Company: got str, expected object

GRAIN
----------------------------------------------------------------------
  ✗ [critical] Grain - Uniqueness: 9984 duplicate (Date, CODE_ISIN) combinations found

IDENTIFIERS
----------------------------------------------------------------------
  ✓ [info    ] Identifiers - CODE_ISIN Null/Empty: No null or empty CODE_ISIN values
  ✓ [info    ] Identifiers - ISIN Format: All ISINs start with MA (valid format)
  ⚠ [warning ] Identifiers - Company Consistency: 16 ISINs have multiple company names

DATES
----------------------------------------------------------------------
  ✓ [info    ] Dates - Null Values: No null dates
  ✓ [

## Step 3: Extract and analyze validation results

In [4]:
# Build results summary
critical_issues = [r for r in validation_report['results'] if r.severity == 'critical' and not r.passed]
warnings = [r for r in validation_report['results'] if r.severity == 'warning' and not r.passed]

print('\n' + '='*70)
print('CRITICAL ISSUES')
print('='*70)
if critical_issues:
    for issue in critical_issues:
        print(f'\n✗ {issue.name}')
        print(f'  {issue.message}')
        if issue.details:
            for k, v in issue.details.items():
                print(f'  - {k}: {v}')
else:
    print('✓ No critical issues found')

print('\n' + '='*70)
print('WARNINGS')
print('='*70)
if warnings:
    for warning in warnings:
        print(f'\n⚠ {warning.name}')
        print(f'  {warning.message}')
        if warning.details:
            for k, v in warning.details.items():
                print(f'  - {k}: {v}')
else:
    print('✓ No warnings found')


CRITICAL ISSUES

✗ Grain - Uniqueness
  9984 duplicate (Date, CODE_ISIN) combinations found
  - duplicate_count: 9984

WARNINGS

⚠ Schema - Data Types
  Date: got datetime64[us], expected datetime64[ns]; CODE_ISIN: got str, expected object; Company: got str, expected object

⚠ Identifiers - Company Consistency
  16 ISINs have multiple company names
  - inconsistent_count: 16

⚠ Prices - Bid-Ask Spread
  97/37042 rows have Bid > Ask
  - inverted_count: 97
  - total: 37042

⚠ Prices - Bid Coverage
  73.4% null Bid values
  - null_percentage: 73.36864541804457

⚠ Prices - Ask Coverage
  68.5% null Ask values
  - null_percentage: 68.49387821579555

⚠ Consistency - Date Coverage
  1730 dates have < 76 companies
  - incomplete_dates: 1730
  - max_companies: 76


## Step 4: Detailed analysis - Data completeness

In [5]:
print('\n' + '='*70)
print('DATA COMPLETENESS ANALYSIS')
print('='*70)

data_cols = ['Cours', 'Bid', 'Ask', 'Volume MC', 'Quantité MC']
completeness_matrix = {}

for col in data_cols:
    completeness = (unified[col].notna().sum() / len(unified) * 100)
    completeness_matrix[col] = completeness
    print(f'\n{col}:')
    print(f'  Data coverage: {completeness:.1f}%')
    print(f'  Non-null values: {unified[col].notna().sum()}/{len(unified)}')
    print(f'  Null values: {unified[col].isna().sum()}')

# Overall coverage
any_data = unified[data_cols].notna().any(axis=1).sum()
all_data = unified[data_cols].notna().all(axis=1).sum()

print(f'\nOVERALL:')
print(f'  Rows with any data: {any_data}/{len(unified)} ({any_data/len(unified)*100:.1f}%)')
print(f'  Rows with complete data: {all_data}/{len(unified)} ({all_data/len(unified)*100:.1f}%)')


DATA COMPLETENESS ANALYSIS

Cours:
  Data coverage: 92.9%
  Non-null values: 131233/141217
  Null values: 9984

Bid:
  Data coverage: 26.6%
  Non-null values: 37608/141217
  Null values: 103609

Ask:
  Data coverage: 31.5%
  Non-null values: 44492/141217
  Null values: 96725

Volume MC:
  Data coverage: 67.8%
  Non-null values: 95718/141217
  Null values: 45499

Quantité MC:
  Data coverage: 67.8%
  Non-null values: 95720/141217
  Null values: 45497

OVERALL:
  Rows with any data: 141217/141217 (100.0%)
  Rows with complete data: 26724/141217 (18.9%)


## Step 5: Detailed analysis - Outlier detection

In [6]:
print('\n' + '='*70)
print('OUTLIER DETECTION')
print('='*70)

# Price outliers using IQR method
for col in ['Cours', 'Bid', 'Ask']:
    data = unified[col].dropna()
    if len(data) > 0:
        q1 = data.quantile(0.25)
        q3 = data.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        
        outliers = ((data < lower) | (data > upper)).sum()
        
        print(f'\n{col}:')
        print(f'  Q1: {q1:.2f}, Q3: {q3:.2f}, IQR: {iqr:.2f}')
        print(f'  Normal range: [{lower:.2f}, {upper:.2f}]')
        print(f'  Outliers: {outliers}/{len(data)} ({outliers/len(data)*100:.1f}%)')
        
        if outliers > 0:
            print(f'  Sample outlier values:')
            outlier_rows = unified[(unified[col] < lower) | (unified[col] > upper)][col].head(5)
            for v in outlier_rows:
                print(f'    - {v:.2f}')


OUTLIER DETECTION

Cours:
  Q1: 130.00, Q3: 1200.00, IQR: 1070.00
  Normal range: [-1475.00, 2805.00]
  Outliers: 11486/131233 (8.8%)
  Sample outlier values:
    - 4000.00
    - 3079.00
    - 3000.00
    - 5000.00
    - 4000.00

Bid:
  Q1: 180.00, Q3: 1275.00, IQR: 1095.00
  Normal range: [-1462.50, 2917.50]
  Outliers: 3504/37608 (9.3%)
  Sample outlier values:
    - 3956.00
    - 6000.00
    - 3800.00
    - 2950.00
    - 4153.00

Ask:
  Q1: 185.00, Q3: 1313.00, IQR: 1128.00
  Normal range: [-1507.00, 3005.00]
  Outliers: 3785/44492 (8.5%)
  Sample outlier values:
    - 4149.00
    - 6500.00
    - 4148.00
    - 4299.00
    - 4149.00


## Step 6: Detailed analysis - Cross-company patterns

In [7]:
print('\n' + '='*70)
print('CROSS-COMPANY PATTERNS')
print('='*70)

# Data coverage by company
print('\nData coverage by company:')
company_coverage = {}
for company in sorted(unified['Company'].unique()):
    company_data = unified[unified['Company'] == company]
    price_coverage = company_data[['Cours', 'Bid', 'Ask']].notna().any(axis=1).sum() / len(company_data) * 100
    volume_coverage = company_data[['Volume MC', 'Quantité MC']].notna().any(axis=1).sum() / len(company_data) * 100
    
    company_coverage[company] = {
        'price': price_coverage,
        'volume': volume_coverage,
        'records': len(company_data)
    }
    
    print(f'  {company:30s}: price {price_coverage:5.1f}% | volume {volume_coverage:5.1f}% | {len(company_data):3d} records')

# Find company with best/worst data coverage
best = max(company_coverage.items(), key=lambda x: x[1]['price'])
worst = min(company_coverage.items(), key=lambda x: x[1]['price'])

print(f'\n  Best price coverage: {best[0]} ({best[1]["price"]:.1f}%)')
print(f'  Worst price coverage: {worst[0]} ({worst[1]["price"]:.1f}%)')


CROSS-COMPANY PATTERNS

Data coverage by company:
  AFMA                          : price 100.0% | volume  91.1% | 1877 records
  AFRIC INDUSTRIES              : price 100.0% | volume   0.0% | 627 records
  AFRIC INDUSTRIES SA           : price 100.0% | volume  67.2% | 1877 records
  AFRIQUIA GAZ                  : price 100.0% | volume  69.1% | 1877 records
  AGMA                          : price 100.0% | volume  26.9% | 1877 records
  AKDITAL                       : price 100.0% | volume  99.9% | 890 records
  ALLIANCES                     : price 100.0% | volume  99.9% | 1877 records
  ALUMINIUM DU MAROC            : price 100.0% | volume  63.1% | 1877 records
  ARADEI CAPITAL                : price 100.0% | volume  94.1% | 1394 records
  ATLANTASANAD                  : price 100.0% | volume  96.9% | 1877 records
  ATTIJARIWAFA BANK             : price 100.0% | volume  99.9% | 1877 records
  AUTO - HALL                   : price 100.0% | volume   0.0% | 627 records
  AUTO HALL     

## Step 7: Recommendations

In [8]:
print('\n' + '='*70)
print('RECOMMENDATIONS')
print('='*70)

recommendations = []

# Based on validation results
if critical_issues:
    recommendations.append(
        'FIX CRITICAL ISSUES BEFORE PROCEEDING'
    )
else:
    recommendations.append(
        'All critical checks passed. Dataset is valid for downstream analysis.'
    )

# Data quality recommendations
if validation_report['critical'] == 0 and validation_report['warnings'] == 0:
    recommendations.append(
        'Dataset quality is excellent. Proceed with market metrics computation.'
    )
elif validation_report['warnings'] > 0:
    recommendations.append(
        'Review warnings before proceeding. Some data quality issues detected.'
    )

# Specific recommendations
avg_price_coverage = sum(v['price'] for v in company_coverage.values()) / len(company_coverage)
if avg_price_coverage < 80:
    recommendations.append(
        f'Average price coverage is {avg_price_coverage:.1f}%. Consider handling missing data in downstream analysis.'
    )

print('\nACTIONABLE RECOMMENDATIONS:')
for i, rec in enumerate(recommendations, 1):
    print(f'{i}. {rec}')

print('\nNEXT STEPS:')
if all_passed:
    print('  1. → Proceed to Notebook 07: Market Metrics')
    print('  2. → Compute capitalization and liquidity metrics')
    print('  3. → Use metrics for dynamic filtering')
else:
    print('  1. → Fix critical issues')
    print('  2. → Re-run validation')
    print('  3. → Investigate root causes')


RECOMMENDATIONS

ACTIONABLE RECOMMENDATIONS:
1. FIX CRITICAL ISSUES BEFORE PROCEEDING
2. Review warnings before proceeding. Some data quality issues detected.

NEXT STEPS:
  1. → Fix critical issues
  2. → Re-run validation
  3. → Investigate root causes


## Step 8: Final validation summary

In [9]:
print('\n' + '='*70)
print('FINAL VALIDATION SUMMARY')
print('='*70)

summary = f"""
INGESTION REPORT:
  - Sheets included: {len(ingest_report['sheets_included'])}
  - Sheets excluded: {len(ingest_report['sheets_excluded'])}
  - Unified records: {ingest_report['unified_records']}
  - Companies: {ingest_report['unified_companies']}
  - Sessions: {ingest_report['unified_sessions']}
  - Variables: {', '.join(ingest_report['unified_variables'])}

QUALITY VALIDATION:
  - Total tests: {validation_report['total_tests']}
  - Passed: {validation_report['passed']}
  - Warnings: {validation_report['warnings']}
  - Critical: {validation_report['critical']}
  - Status: {'✓ PASS' if all_passed else '✗ FAIL'}

DATA COMPLETENESS:
  - Average price coverage: {avg_price_coverage:.1f}%
  - Rows with any data: {any_data}/{len(unified)} ({any_data/len(unified)*100:.1f}%)
  - Rows with complete data: {all_data}/{len(unified)} ({all_data/len(unified)*100:.1f}%)

NEXT PHASE:
  → Notebook 07: Market Metrics Computation
"""

print(summary)


FINAL VALIDATION SUMMARY

INGESTION REPORT:
  - Sheets included: 5
  - Sheets excluded: 2
  - Unified records: 141217
  - Companies: 77
  - Sessions: 1877
  - Variables: Ask, Bid, Cours, Quantité MC, Volume MC

QUALITY VALIDATION:
  - Total tests: 13
  - Passed: 6
  - Warnings: 6
  - Critical: 1
  - Status: ✗ FAIL

DATA COMPLETENESS:
  - Average price coverage: 100.0%
  - Rows with any data: 141217/141217 (100.0%)
  - Rows with complete data: 26724/141217 (18.9%)

NEXT PHASE:
  → Notebook 07: Market Metrics Computation

